# Neo-Hookean 2D Membrane Shell (Edge-Vector Symbolic Verification)

`src/backends/cuda/finite_element/constitutions/sym/neo_hookean_shell_2d.inl` is maintained
as a hand-written analytical implementation directly in edge vectors $e_1 = x_1 - x_0$,
$e_2 = x_2 - x_0$ with $\det(A) = \|e_1 \times e_2\|^2$.

**Why not generate directly from `X` with SymPy `A.det()`?**
Calling `A.det()` symbolically on raw world coordinates $X \in \mathbb{R}^9$ expands
$\det(A)$ into 64 degree-4 monomials of magnitude $O(|X|^4)$, whereas
$\det(A) = 4\,\mathrm{Area}^2 = O(h^4)$. For fine shells simulated away from the origin,
the $O((|X|/h)^4)$ cancellation destroys FP64 accuracy and injects non-smooth jitter into
line-search energy evaluations.

This notebook symbolically verifies that the edge-vector analytical formulas in
`neo_hookean_shell_2d.inl` match exact SymPy differentiation, without overwriting the `.inl` file.

In [ ]:
import sys
sys.path.append('../')
from SymEigen import Eigen, VecDiff
from sympy import Matrix, Rational, cancel, expand, eye, log, symbols

lmbd, mu = symbols('lambda mu')
ib00, ib01, ib11 = symbols('ib00 ib01 ib11')
IB = Matrix([[ib00, ib01], [ib01, ib11]])

# Edge-vector coordinates D = [e1; e2] = [x1 - x0; x2 - x0]
D = Eigen.Vector('D', 6)
e1 = D[0:3, 0]
e2 = D[3:6, 0]

a00 = e1.dot(e1)
a01 = e1.dot(e2)
a11 = e2.dot(e2)
A = Matrix([[a00, a01], [a01, a11]])

detA = e1.cross(e2).dot(e1.cross(e2))
assert expand(detA - A.det()) == 0

detIB = IB.det()
log_J2 = log(detA * detIB)
tr = IB[0, 0] * a00 + (IB[0, 1] + IB[1, 0]) * a01 + IB[1, 1] * a11
E = Rational(1, 8) * lmbd * log_J2**2 + Rational(1, 2) * mu * (tr - log_J2 - 2)

In [ ]:
# Analytical gradient dE/dD matching sym/neo_hookean_shell_2d.inl
invA00 = a11 / detA
invA01 = -a01 / detA
invA11 = a00 / detA
alpha = Rational(1, 2) * lmbd * log_J2 - mu
s00 = mu * IB[0, 0] + alpha * invA00
s01 = mu * ib01 + alpha * invA01
s11 = mu * IB[1, 1] + alpha * invA11

g1 = s00 * e1 + s01 * e2
g2 = s01 * e1 + s11 * e2
G_D = Matrix.vstack(g1, g2)

G_sym = VecDiff(E, D)
for i in range(6):
    assert cancel(G_D[i] - G_sym[i]) == 0, f'Gradient mismatch at entry {i}'
print('Exact symbolic gradient dE/dD verified.')

In [ ]:
# Analytical Hessian d^2E/dD^2 matching sym/neo_hookean_shell_2d.inl
q0 = invA00 * e1 + invA01 * e2
q1 = invA01 * e1 + invA11 * e2
P = e1 * q0.T + e2 * q1.T
I3 = eye(3)

H11 = s00 * I3 + (lmbd - alpha) * (q0 * q0.T) - (alpha * invA00) * P
H12 = s01 * I3 + lmbd * (q0 * q1.T) - alpha * (invA01 * P + q1 * q0.T)
H22 = s11 * I3 + (lmbd - alpha) * (q1 * q1.T) - (alpha * invA11) * P
H_D = Matrix.vstack(Matrix.hstack(H11, H12), Matrix.hstack(H12.T, H22))

H_sym = VecDiff(G_D, D)
for i in range(6):
    for j in range(6):
        assert cancel(H_D[i, j] - H_sym[i, j]) == 0, f'Hessian mismatch at ({i}, {j})'
print('Exact symbolic Hessian d^2E/dD^2 verified.')

In [ ]:
# Verify exact chain rule from D = J_DX * X to X = [x0; x1; x2]
J_DX = Matrix.vstack(
    Matrix.hstack(-I3, I3, Matrix.zeros(3, 3)),
    Matrix.hstack(-I3, Matrix.zeros(3, 3), I3),
)
G_9 = Matrix.vstack(-(g1 + g2), g1, g2)
assert (G_9 - J_DX.T * G_D).applyfunc(expand) == Matrix.zeros(9, 1)

H21 = H12.T
H01 = -(H11 + H21)
H02 = -(H12 + H22)
H00 = -(H01 + H02)
H_9 = Matrix.vstack(
    Matrix.hstack(H00, H01, H02),
    Matrix.hstack(H01.T, H11, H12),
    Matrix.hstack(H02.T, H21, H22),
)
assert (H_9 - J_DX.T * H_D * J_DX).applyfunc(expand) == Matrix.zeros(9, 9)
print('Exact 9-DOF gradient and Hessian chain-rule mapping verified.')